# 14 — Ridge β extraction and brain panels

In [ ]:
import os
os.environ.setdefault("PYVISTA_OFF_SCREEN", "true")

from pathlib import Path
import sys

PROJECT_ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                    if (p / "analysis" / "lib").exists())
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from joblib import Parallel, delayed

from analysis.lib.data import (load_label, restricted_pool,
                               build_demographics_matrix)
from analysis.lib.pipelines import make_base_learner
from analysis.lib.cv import pooled_cv
from analysis.lib.atlases import xtract_data_dict

DATA_DIR   = PROJECT_ROOT / "data"
LACUNA_DIR = DATA_DIR / "lacuna" / "METAVCI_PSCI_LACUNA" / "derivatives"
BETAS_DIR  = PROJECT_ROOT / "outputs" / "feature_importance" / "ridge_betas"
FIG_BASE   = PROJECT_ROOT / "outputs" / "feature_importance" / "figures"
MOD_OUT_DIRS = {
    "sLNM":   FIG_BASE / "cells_fig2_ridge_betas",
    "fLNM":   FIG_BASE / "cells_suppl_betas_flnm",
    "Lesion": FIG_BASE / "cells_suppl_betas_lesion",
}
NETTEKOVEN32 = (PROJECT_ROOT / "outputs" / "feature_importance"
                / "yabplot_custom_atlases" / "nettekoven32")
BETAS_DIR.mkdir(parents=True, exist_ok=True)
for d in MOD_OUT_DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

CLIN_PATH = DATA_DIR / "tabular" / "clinical_data.csv"
assert CLIN_PATH.exists(), f"missing {CLIN_PATH} — see data/README.md"

# (short tag, full target column) — order matches the manuscript
TARGETS = [
    ("PSCI",     "cognition-PSCI_present"),
    ("aef",      "cognition-aef_domain_score"),
    ("ps",       "cognition-ps_domain_score"),
    ("language", "cognition-language_domain_score"),
    ("verbmem",  "cognition-verbalmemory_domain_score"),
    ("vsfunc",   "cognition-vsfunctions_domain_score"),
    ("vsmem",    "cognition-vsmemory_domain_score"),
]

In [ ]:
# Ridge β extraction across pooled CV folds (cached per target CSV)
clinical_df = pd.read_csv(CLIN_PATH, index_col=0)
acute_pool = restricted_pool(LACUNA_DIR, clinical_df, all_of=["acuteinfarct"])
clinical = clinical_df.loc[acute_pool]
X_demo, demo_names, _, _, _ = build_demographics_matrix(clinical, exclude_cohort=False)
n_demo = X_demo.shape[1]

MOD_FEATURES = {
    "sLNM":   load_label(LACUNA_DIR, "acuteinfarct", "snm", "disconnectionpct").reindex(acute_pool),
    "fLNM":   load_label(LACUNA_DIR, "acuteinfarct", "fnm", "tmap").reindex(acute_pool),
    "Lesion": load_label(LACUNA_DIR, "acuteinfarct", "rd",  "damagepct").reindex(acute_pool),
}
parcels = list(MOD_FEATURES["sLNM"].columns)
print(f"acute pool {len(clinical)} | {len(parcels)} parcels | n_demo {n_demo}")


def _fit_one_fold(target, train_idx, X_total):
    y = clinical[target].values
    pipe = make_base_learner(n_demo, X_total.shape[1])
    pipe.fit(X_total[train_idx], y[train_idx])
    coef = pipe.named_steps["clf"].coef_.ravel()
    return coef[n_demo:]   # imaging block only


for modality, feat_df in MOD_FEATURES.items():
    X_total = np.hstack([X_demo, feat_df.values])
    for short, target in TARGETS:
        csv = BETAS_DIR / f"{modality}_{target}.csv"
        if csv.exists():
            print(f"[skip] {csv.name} exists")
            continue
        y_series = clinical[target]
        valid_idx = np.where(y_series.notna().values)[0]
        if len(valid_idx) < 50:
            print(f"[skip] {target}: only {len(valid_idx)} valid")
            continue
        cv = pooled_cv(seed=42)
        y_valid = y_series.values[valid_idx]
        abs_splits = [(valid_idx[tr], valid_idx[te])
                      for tr, te in cv.split(np.zeros((len(valid_idx), 1)), y_valid)]
        print(f"fitting {modality} / {target} ({len(abs_splits)} folds)")
        coefs = np.asarray(Parallel(n_jobs=-1)(
            delayed(_fit_one_fold)(target, tr, X_total) for tr, _ in abs_splits))
        pd.DataFrame({
            "parcel":    parcels,
            "mean_beta": coefs.mean(axis=0),
            "se_beta":   coefs.std(axis=0, ddof=1) / np.sqrt(len(coefs)),
            "n_folds":   coefs.shape[0],
        }).set_index("parcel").to_csv(csv)
        print(f"  saved {csv.name}")

In [ ]:
# Shared symmetric vmax (per modality) + parcel-section partition
MOD_BETAS, MOD_VMAX = {}, {}
for modality, out_dir in MOD_OUT_DIRS.items():
    per_target = {}
    for short, full in TARGETS:
        df = pd.read_csv(BETAS_DIR / f"{modality}_{full}.csv", index_col=0)
        per_target[short] = df["mean_beta"]
    betas = pd.DataFrame(per_target)
    vmax = float(np.percentile(np.abs(betas.values), 95))
    print(f"{modality}: betas {betas.shape}, shared vmax (95%ile) = {vmax:.4f}")
    pd.DataFrame({"vmax": [vmax]}).to_csv(out_dir / "shared_vmax.csv", index=False)
    betas.to_csv(out_dir / "mean_betas_matrix.csv")
    MOD_BETAS[modality], MOD_VMAX[modality] = betas, vmax

betas = MOD_BETAS["sLNM"]


def partition(parcels):
    cortex = [p for p in parcels if p.startswith("7Networks_")]
    sub    = [p for p in parcels if (not p.startswith("7Networks_"))
              and (p.endswith("-lh") or p.endswith("-rh"))]
    tract  = [p for p in parcels if p not in cortex and p not in sub
              and any(c.isupper() for c in p) and " " in p]
    ceb    = [p for p in parcels if p not in cortex and p not in sub and p not in tract]
    return {"cortex": cortex, "subcortex": sub, "tracts": tract, "cerebellum": ceb}


parts = partition(list(betas.index))
print({k: len(v) for k, v in parts.items()})

In [ ]:
# Render per-target × per-section brain PNGs (yabplot; cached by existence)
import pyvista as pv
pv.OFF_SCREEN = True
if hasattr(pv, "start_xvfb") and "DISPLAY" not in os.environ:
    pv.start_xvfb()

import yabplot as yab
import yabplot.plotting as _yp
_yp.add_colorbars = lambda *a, **kw: None   # single shared colorbar added at composition

import yabplot.scene as _yab_scene
_orig_finalize = _yab_scene.finalize_plot
def _patched_finalize(plotter, export_path, display_type):
    plotter.off_screen = True
    plotter.render()
    return _orig_finalize(plotter, export_path, display_type)
_yab_scene.finalize_plot = _patched_finalize
_yp.finalize_plot = _patched_finalize


def render_section(section, target_short, beta_series, out_dir, vmax):
    out_png = out_dir / f"{target_short}_{section}.png"
    if out_png.exists():
        print(f"  [skip] {out_png.name} exists")
        return
    data = {p: float(beta_series[p]) for p in parts[section] if pd.notna(beta_series[p])}
    if not data:
        print(f"  [skip] no data for {target_short}/{section}")
        return
    kwargs = dict(data=data, cmap="RdBu_r", vminmax=[-vmax, vmax],
                  display_type="static", export_path=str(out_png))
    if section == "cortex":
        yab.plot_cortical(atlas="schaefer100",
                          views=["left_lateral", "right_lateral"],
                          bmesh="inflated", figsize=(1400, 600), **kwargs)
    elif section == "subcortex":
        yab.plot_subcortical(atlas="tian2020_s1",
                             views=["left_medial", "right_medial"],
                             figsize=(700, 500), **kwargs)
    elif section == "tracts":
        data_x = xtract_data_dict(data)
        if not data_x:
            print(f"  [skip] xtract empty {target_short}")
            return
        kwargs["data"] = data_x
        yab.plot_tracts(atlas="xtract_tiny", views=["superior"],
                        bmesh="pial", bmesh_alpha=0.1, figsize=(700, 700), **kwargs)
    elif section == "cerebellum":
        assert NETTEKOVEN32.exists(), f"missing custom atlas {NETTEKOVEN32}"
        yab.plot_subcortical(custom_atlas_path=str(NETTEKOVEN32),
                             views=["posterior"],
                             bmesh_alpha=0.05, figsize=(700, 700), **kwargs)
    print(f"  saved {out_png.name}")


for modality, out_dir in MOD_OUT_DIRS.items():
    for short, _ in TARGETS:
        for section in ("cortex", "subcortex", "tracts", "cerebellum"):
            render_section(section, short, MOD_BETAS[modality][short],
                           out_dir, MOD_VMAX[modality])
print(f"ridge-beta brain panels at {FIG_BASE}")